# ML-04 — Search Intelligence Data Contract

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandini3206/flyrank-ml-workspace/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### Unit of Analysis
The unit of analysis is a single pseudonymized content item (uniquely identified by `content_hash_id` and `client_hash_id`) during a specific decision month.

### Time Window Setup
We utilize a strict split of the temporal data to ensure prediction-time discipline:
- **Decision Month / Feature Window:** March 2026 (`month = '2026-03'`). All features are aggregated or computed over this month.
- **Outcome / Label Window:** April 2026 (`month = '2026-04'`). The outcome label `is_declining` is evaluated using April 2026 search performance compared to March 2026.

This mid-panel month setup represents the actual operational flow where a model stands at the end of March 2026 and predicts upcoming decline in April 2026, avoiding any future lookahead and ensuring realistic evaluation.

In [ ]:
import os
import getpass
import duckdb

# Setup Hugging Face token
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

# Fetch basic stats about the selected time windows
print("Verifying time partitions and rows...")
df_windows = con.execute(f"""
    SELECT month, MIN(report_date) AS start_date, MAX(report_date) AS end_date, COUNT(*) AS daily_records
    FROM {TABLES['fact_daily']}
    WHERE month IN ('2026-03', '2026-04')
    GROUP BY month
    ORDER BY month
""").df()
print(df_windows)

## 2. Fields: feature / label / context / excluded

### Field Classification (Lane 2 - Refresh / Content Opportunity Scoring)

We classify all fields we touch into exactly four buckets:

1. **Features (exactly 5 features):**
   - `mar_impressions`: Sum of GSC impressions in March 2026. Represents total search visibility/demand. Available at decision time because GSC reports are finalized prior to April 1, 2026.
   - `mar_clicks`: Sum of GSC clicks in March 2026. Represents active search click-through volume. Available at decision time.
   - `mar_avg_position`: Weighted average of daily average GSC position in March 2026 (weighted by GSC impressions). Represents rank stability. Available at decision time.
   - `mar_ga4_sessions_per_day`: Average GSC session count per day in March 2026, filtered specifically on rows where `ga4_data_available IS TRUE` is true. Represents direct on-page engagement. Available at decision time.
   - `content_age_days`: Age of the content in days at the decision point (March 31, 2026), computed as `DATEDIFF('day', content_created_date, '2026-03-31')`. Available at decision time.

2. **Label / Proxy:**
   - `is_declining`: Target binary outcome defined as `1` if future (April 2026) impressions fall below 80% of current (March 2026) impressions (`apr_impressions < 0.8 * mar_impressions`), otherwise `0`.

3. **Context:**
   - `client_hash_id`: Scrambled client ID used strictly for cohort grouping/splitting.
   - `content_hash_id`: Scrambled content ID used for joining and unique identification.

4. **Excluded (with one-line why each):**
   - `month`: Part of the partitioning, excluded to prevent the model from learning specific temporal indices.
   - `is_active`, `is_published`, `is_deleted`: Admin states that do not provide search performance intelligence.
   - `apr_impressions` / `apr_clicks`: Future metrics from the outcome window, which directly leak the label.
   - `trend_pct` / `trend_direction`: Retrospectively calculated trajectory features that leak future outcome trends.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

print("Compiling features and label dataset from the warehouse...")
dataset = con.execute(f"""
    WITH mar_perf AS (
        SELECT 
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS mar_impressions,
            SUM(gsc_clicks) AS mar_clicks,
            CASE 
                WHEN SUM(gsc_impressions) > 0 
                THEN SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions) 
                ELSE NULL 
            END AS mar_avg_position,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS mar_ga4_sessions,
            COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) AS mar_ga4_days
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
        GROUP BY 1, 2
    ),
    apr_perf AS (
        SELECT 
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS apr_impressions
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-04'
        GROUP BY 1, 2
    )
    SELECT 
        m.client_hash_id,
        m.content_hash_id,
        m.mar_impressions,
        m.mar_clicks,
        m.mar_avg_position,
        CASE WHEN m.mar_ga4_days > 0 THEN m.mar_ga4_sessions / m.mar_ga4_days ELSE 0.0 END AS mar_ga4_sessions_per_day,
        DATEDIFF('day', CAST(c.content_created_date AS DATE), CAST('2026-03-31' AS DATE)) AS content_age_days,
        COALESCE(a.apr_impressions, 0) AS apr_impressions, -- Leaked feature
        CASE 
            WHEN a.apr_impressions IS NULL THEN 1 
            WHEN a.apr_impressions < 0.8 * m.mar_impressions THEN 1
            ELSE 0
        END AS is_declining
    FROM mar_perf m
    LEFT JOIN apr_perf a 
        ON m.client_hash_id = a.client_hash_id 
       AND m.content_hash_id = a.content_hash_id
    LEFT JOIN {TABLES['dim_content']} c 
        ON m.content_hash_id = c.content_hash_id
    WHERE m.mar_impressions >= 100
""").df()

print(f"Dataset ready: {len(dataset):,} rows")
dataset = dataset.dropna(subset=['mar_avg_position', 'content_age_days'])
print(f"Cleaned dataset: {len(dataset):,} rows")

X_honest_cols = ['mar_impressions', 'mar_clicks', 'mar_avg_position', 'mar_ga4_sessions_per_day', 'content_age_days']
X_leaked_cols = X_honest_cols + ['apr_impressions']
y = dataset['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(dataset, y, test_size=0.25, random_state=42, stratify=y)

print("\n--- LEAKAGE EXPERIMENT ---")
model_leaked = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_leaked.fit(X_tr[X_leaked_cols], y_tr)
preds_leaked = model_leaked.predict(X_te[X_leaked_cols])
auc_leaked = roc_auc_score(y_te, model_leaked.predict_proba(X_te[X_leaked_cols])[:, 1])
print(f"Leaked Model ROC AUC: {auc_leaked:.4f}")
print(classification_report(y_te, preds_leaked, digits=4))
print("Leaked Feature Importances:")
for f, imp in zip(X_leaked_cols, model_leaked.feature_importances_):
    print(f"  {f}: {imp:.4f}")

print("\n--- HONEST MODEL (LEAKED FEATURE REMOVED) ---")
model_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_honest.fit(X_tr[X_honest_cols], y_tr)
preds_honest = model_honest.predict(X_te[X_honest_cols])
auc_honest = roc_auc_score(y_te, model_honest.predict_proba(X_te[X_honest_cols])[:, 1])
print(f"Honest Model ROC AUC: {auc_honest:.4f}")
print(classification_report(y_te, preds_honest, digits=4))
print("Honest Feature Importances:")
for f, imp in zip(X_honest_cols, model_honest.feature_importances_):
    print(f"  {f}: {imp:.4f}")

## 3. Verify it with queries (grain, counts, missing values, windows)

### Query Verification of the Data Contract

To make sure our contract is grounded in evidence and matches the warehouse facts, we execute exactly three verification queries below:

1. **Query 1: Grain Verification.** Verify that `client_hash_id` + `content_hash_id` is unique at our unit of analysis (March 2026 performance level) and that duplicate rows do not exist.
2. **Query 2: Volume and Date Coverage Verification.** Query total rows, distinct clients, distinct content items, and the exact min/max `report_date` for March 2026 and April 2026 to ensure window boundaries hold perfectly.
3. **Query 3: Missingness and Flag Verification.** Check the missingness rate of GSC avg position and GA4-based daily sessions (restricted to `ga4_data_available IS TRUE`) to confirm active engagement tracking presence without category-signal injection.

In [ ]:
print("--- QUERY 1: GRAIN UNIQUE VERIFICATION ---")
grain_df = con.execute(f"""
    SELECT client_hash_id, content_hash_id, COUNT(*) AS num_records
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
    HAVING COUNT(*) > 31
    LIMIT 5
""").df()
print("Duplicate grains >31 days in March (expected empty):", grain_df.shape[0])

print("\n--- QUERY 2: VOLUME & BOUNDARIES VERIFICATION ---")
vol_df = con.execute(f"""
    SELECT 
        month,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS active_clients,
        COUNT(DISTINCT content_hash_id) AS active_pages,
        COUNT(*) AS row_count
    FROM {TABLES['fact_daily']}
    WHERE month IN ('2026-03', '2026-04')
    GROUP BY month
    ORDER BY month
""").df()
print(vol_df)

print("\n--- QUERY 3: MISSINGNESS RATES & FLAG VERIFICATION ---")
miss_df = con.execute(f"""
    SELECT 
        COUNT(*) AS total_daily_rows,
        AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0.0 END) AS gsc_pos_null_rate,
        AVG(CASE WHEN ga4_data_available IS TRUE AND ga4_sessions IS NULL THEN 1.0 ELSE 0.0 END) AS ga4_sess_null_rate_active_tracking
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
print(miss_df)

## 4. Data limits

### Inherent Data Limits

1. **Unbalanced History Depth:** Clients have wildly different historical starting points (different `gsc_data_start` and `ga4_data_start`). Models must be evaluated on active windows.
2. **GSC-Only Early Rows:** Prior to `ga4_data_start`, all GA4-related columns are zero-filled. Direct modeling of zeroes as 'no traffic' would lead to severe bias unless filtered on `ga4_data_available IS TRUE`.
3. **Window Overlaps and Trailing Metrics:** Trailing-90-day metrics can overlap with the target period if not strictly aligned, leading to silent leakage. By using separate partitions (March vs. April 2026), we prevent any overlap.
4. **Scrambled Entities:** IDs are completely pseudonymized, so semantic content or client vertical cannot be inferred. Predictions must rely purely on numeric performance signals.

In [ ]:
# Query showcasing unbalanced history of clients
print("Querying client GSC vs GA4 start dates to demonstrate limits:")
clients_history = con.execute(f"""
    SELECT 
        COUNT(*) AS total_clients,
        MIN(gsc_data_start) AS earliest_gsc_start,
        MAX(gsc_data_start) AS latest_gsc_start,
        MIN(ga4_data_start) AS earliest_ga4_start,
        MAX(ga4_data_start) AS latest_ga4_start
    FROM {TABLES['dim_clients']}
""").df()
print(clients_history)

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.